# UJIIndoorLoc — preparación y clustering autocontenidos


## 2. Importaciones y rutas


In [1]:
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler

try:
    from IPython.display import display
except ImportError:
    display = print


# Si Jupyter se inicia fuera de la carpeta del paquete, escribe aquí su ruta.
# Ejemplo: PROJECT_ROOT = Path(r"C:/TFM/notebooks_autocontenidos_sin_core")
PROJECT_ROOT = None

cwd = Path.cwd().resolve()
root_candidates = [cwd, cwd.parent, cwd.parent.parent]
if PROJECT_ROOT is not None:
    ROOT = Path(PROJECT_ROOT).expanduser().resolve()
else:
    ROOT = next(
        (
            candidate
            for candidate in root_candidates
            if sum((candidate / name).is_dir() for name in ["TUT", "TUJI1", "UJIIndoor", "SOD"])
            >= 2
        ),
        cwd,
    )

SEED = 42
TARGET_COLUMNS = ["TARGET_X_M", "TARGET_Y_M"]
print("Raíz utilizada:", ROOT)


Raíz utilizada: /home/coder/Indoor/Notebooks


## 3. Lectura y preprocesado RSSI

Se normalizan los nombres de columnas, se detectan automáticamente WAP/MAC
y el escalador se ajusta solo con la partición de entrenamiento.


In [2]:
def natural_key(text: str) -> List[object]:
    return [int(piece) if piece.isdigit() else piece for piece in re.split(r"(\d+)", text)]


def find_file_case_insensitive(filename: str, directories: Sequence[Path]) -> Path:
    """Busca un nombre sin depender de mayusculas/minusculas."""
    checked: List[str] = []
    target = filename.casefold()
    for directory in directories:
        directory = Path(directory)
        checked.append(str(directory / filename))
        if not directory.exists():
            continue
        direct = directory / filename
        if direct.exists():
            return direct.resolve()
        for child in directory.iterdir():
            if child.is_file() and child.name.casefold() == target:
                return child.resolve()
    raise FileNotFoundError(
        f"No se encontro {filename}. Rutas comprobadas:\n- " + "\n- ".join(checked)
    )


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [str(c).strip().upper() for c in out.columns]
    return out


def detect_rssi_columns(df: pd.DataFrame) -> List[str]:
    cols = [c for c in df.columns if re.fullmatch(r"(?:WAP|MAC)\d+", str(c).upper())]
    cols = sorted(cols, key=natural_key)
    if not cols:
        raise ValueError("No se detectaron columnas RSSI WAPnnn o MACnnn.")
    return cols


class RSSIPreprocessor:
    """Imputa ausencias, estandariza RSSI con train y anade mascara de deteccion."""

    def __init__(self, missing_value: float = 100.0, fill_value: float = -110.0, use_mask: bool = True):
        self.missing_value = float(missing_value)
        self.fill_value = float(fill_value)
        self.use_mask = bool(use_mask)
        self.scaler = StandardScaler()
        self.columns: List[str] = []

    def _clean(self, df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
        raw = df[self.columns].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
        observed = np.isfinite(raw) & (raw != self.missing_value)
        clean = np.where(observed, raw, self.fill_value).astype(np.float32)
        return clean, observed.astype(np.float32)

    def fit(self, df: pd.DataFrame, columns: Sequence[str]) -> "RSSIPreprocessor":
        self.columns = list(columns)
        clean, _ = self._clean(df)
        self.scaler.fit(clean)
        return self

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        clean, mask = self._clean(df)
        scaled = self.scaler.transform(clean).astype(np.float32)
        if self.use_mask:
            return np.concatenate([scaled, mask], axis=1).astype(np.float32)
        return scaled

    def fit_transform(self, df: pd.DataFrame, columns: Sequence[str]) -> np.ndarray:
        return self.fit(df, columns).transform(df)


## 4. Coordenadas y división de posiciones


In [3]:
def _webmercator_latitude_radians(y: np.ndarray) -> np.ndarray:
    radius = 6378137.0
    return 2.0 * np.arctan(np.exp(np.asarray(y, dtype=float) / radius)) - np.pi / 2.0


def add_uji_metric_targets(
    df: pd.DataFrame,
    mode: str = "epsg3857_to_epsg25830",
) -> pd.DataFrame:
    """Crea objetivos metricos para UJIIndoorLoc.

    El modo recomendado interpreta LONGITUDE/LATITUDE como EPSG:3857 y las
    reproyecta a ETRS89 / UTM 30N (EPSG:25830). Si el CRS de una copia del
    dataset se conoce y es distinto, debe cambiarse explicitamente.
    """
    out = df.copy()
    x = pd.to_numeric(out["LONGITUDE"], errors="raise").to_numpy(dtype=float)
    y = pd.to_numeric(out["LATITUDE"], errors="raise").to_numpy(dtype=float)
    mode_norm = mode.strip().lower()

    if mode_norm == "epsg3857_to_epsg25830":
        try:
            from pyproj import Transformer

            transformer = Transformer.from_crs("EPSG:3857", "EPSG:25830", always_xy=True)
            x_m, y_m = transformer.transform(x, y)
            metric_crs = "EPSG:25830_FROM_EPSG:3857"
        except ImportError:
            # Aproximacion local conforme: Web Mercator exagera ambas direcciones
            # aproximadamente por sec(latitud).
            lat0 = float(np.nanmean(_webmercator_latitude_radians(y)))
            factor = math.cos(lat0)
            # No centramos por separado cada split: eso desplazaria train,
            # validacion y test a origenes incompatibles. El escalador de
            # objetivos posterior se ajusta exclusivamente con train.
            x_m = x * factor
            y_m = y * factor
            metric_crs = f"LOCAL_GROUND_APPROX_COS_LAT_{factor:.8f}"
            warnings.warn(
                "pyproj no esta instalado: se usa correccion local cos(lat). "
                "Instala pyproj para obtener EPSG:25830 exacto.",
                RuntimeWarning,
            )
    elif mode_norm == "projected_as_metres":
        x_m, y_m = x, y
        metric_crs = "ORIGINAL_PROJECTED_UNITS_ASSUMED_METRES"
    else:
        raise ValueError(
            "UJI_METRIC_MODE debe ser 'epsg3857_to_epsg25830' o 'projected_as_metres'."
        )

    out["TARGET_X_M"] = np.asarray(x_m, dtype=float)
    out["TARGET_Y_M"] = np.asarray(y_m, dtype=float)
    out["METRIC_CRS"] = metric_crs
    return out


def position_key(df: pd.DataFrame, columns: Sequence[str]) -> pd.Series:
    return df[list(columns)].astype(str).agg("|".join, axis=1)


def split_uji_training_by_position(
    df: pd.DataFrame,
    position_columns: Sequence[str],
    val_size: float = 0.15,
    seed: int = SEED,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    work = df.copy()
    groups = position_key(work, position_columns)
    splitter = GroupShuffleSplit(n_splits=1, test_size=val_size, random_state=seed)
    train_idx, val_idx = next(splitter.split(work, groups=groups))
    train_out = work.iloc[train_idx].reset_index(drop=True)
    val_out = work.iloc[val_idx].reset_index(drop=True)
    overlap = set(position_key(train_out, position_columns)) & set(position_key(val_out, position_columns))
    if overlap:
        raise AssertionError("La division UJI conserva posiciones compartidas entre train y val.")
    return train_out, val_out


## 5. Guardado y router RSSI→clúster

Los identificadores de fila permiten unir cada partición con sus rutas.
KMeans se ajusta con las posiciones de train; un Extra Trees aprende a
reproducir esas zonas desde RSSI para validación y test.


### 5.1. Guardado de las tres particiones


In [4]:
def _assign_row_ids(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    out = df.reset_index(drop=True).copy()
    out.insert(0, "ROW_ID", [f"{prefix}_{i:07d}" for i in range(len(out))])
    return out


def save_base_splits(
    train: pd.DataFrame,
    val: pd.DataFrame,
    test: pd.DataFrame,
    output_dir: Path,
    prefix: str,
) -> Dict[str, Path]:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    split_frames = {
        "train": _assign_row_ids(train, f"{prefix}_train"),
        "val": _assign_row_ids(val, f"{prefix}_val"),
        "test": _assign_row_ids(test, f"{prefix}_test"),
    }
    paths: Dict[str, Path] = {}
    for split, frame in split_frames.items():
        path = output_dir / f"{prefix}_{split}.csv"
        frame.to_csv(path, index=False)
        paths[split] = path
    return paths


def read_base_splits(output_dir: Path, prefix: str) -> Dict[str, pd.DataFrame]:
    output_dir = Path(output_dir)
    return {
        split: pd.read_csv(output_dir / f"{prefix}_{split}.csv")
        for split in ["train", "val", "test"]
    }


### 5.2. Entrenamiento del router y auditoría


In [5]:
def fit_rssi_routing(
    train: pd.DataFrame,
    val: pd.DataFrame,
    test: pd.DataFrame,
    rssi_columns: Sequence[str],
    k_values: Sequence[int],
    seed: int = SEED,
    n_estimators: int = 200,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Define zonas con XY de train y aprende RSSI -> zona para val/test.

    CLUSTER_ORACLE solo se conserva para diagnostico. La columna CLUSTER que
    consumen los modelos es predicha por RSSI en validacion y test.
    """
    pre = RSSIPreprocessor(use_mask=True).fit(train, rssi_columns)
    x_train = pre.transform(train)
    x_val = pre.transform(val)
    x_test = pre.transform(test)
    coords_train = train[TARGET_COLUMNS].to_numpy(dtype=float)
    unique_coords = np.unique(coords_train, axis=0)

    route_parts: List[pd.DataFrame] = []
    diagnostics: List[Dict[str, object]] = []
    for k in sorted(set(int(v) for v in k_values)):
        if k < 2 or k > len(unique_coords):
            continue
        kmeans = KMeans(n_clusters=k, random_state=seed, n_init=20)
        kmeans.fit(unique_coords)
        oracle = {
            "train": kmeans.predict(train[TARGET_COLUMNS].to_numpy(dtype=float)),
            "val": kmeans.predict(val[TARGET_COLUMNS].to_numpy(dtype=float)),
            "test": kmeans.predict(test[TARGET_COLUMNS].to_numpy(dtype=float)),
        }
        gate = ExtraTreesClassifier(
            n_estimators=n_estimators,
            min_samples_leaf=2,
            max_features="sqrt",
            class_weight="balanced",
            random_state=seed + k,
            n_jobs=-1,
        )
        gate.fit(x_train, oracle["train"])
        predicted = {
            "train": oracle["train"],
            "val": gate.predict(x_val),
            "test": gate.predict(x_test),
        }
        probabilities = {
            "train": np.ones(len(train), dtype=float),
            "val": np.max(gate.predict_proba(x_val), axis=1),
            "test": np.max(gate.predict_proba(x_test), axis=1),
        }
        for split, frame in [("train", train), ("val", val), ("test", test)]:
            route_parts.append(
                pd.DataFrame(
                    {
                        "ROW_ID": frame["ROW_ID"].astype(str).to_numpy(),
                        "SPLIT": split,
                        "N_CLUSTERS": k,
                        "CLUSTER": predicted[split].astype(int),
                        "CLUSTER_ORACLE": oracle[split].astype(int),
                        "GATE_CONFIDENCE": probabilities[split].astype(float),
                    }
                )
            )
        diagnostics.append(
            {
                "N_CLUSTERS": k,
                "VAL_GATE_ACCURACY": float(accuracy_score(oracle["val"], predicted["val"])),
                "TEST_GATE_ACCURACY_DIAGNOSTIC_ONLY": float(
                    accuracy_score(oracle["test"], predicted["test"])
                ),
                "VAL_MEAN_CONFIDENCE": float(np.mean(probabilities["val"])),
                "TEST_MEAN_CONFIDENCE": float(np.mean(probabilities["test"])),
                "N_TRAIN_POSITIONS": int(len(unique_coords)),
            }
        )
    if not route_parts:
        raise ValueError("No se genero ninguna configuracion de clustering.")
    return pd.concat(route_parts, ignore_index=True), pd.DataFrame(diagnostics)


def save_routing(
    routes: pd.DataFrame,
    diagnostics: pd.DataFrame,
    output_dir: Path,
    prefix: str,
) -> Tuple[Path, Path]:
    output_dir = Path(output_dir)
    route_path = output_dir / f"{prefix}_routes.csv"
    diagnostics_path = output_dir / f"{prefix}_routing_diagnostics.csv"
    routes.to_csv(route_path, index=False)
    diagnostics.to_csv(diagnostics_path, index=False)
    return route_path, diagnostics_path


def validate_splits(
    train: pd.DataFrame,
    val: pd.DataFrame,
    test: pd.DataFrame,
    position_columns: Sequence[str],
) -> Dict[str, object]:
    ids = [set(frame["ROW_ID"].astype(str)) for frame in [train, val, test]]
    if ids[0] & ids[1] or ids[0] & ids[2] or ids[1] & ids[2]:
        raise AssertionError("ROW_ID se solapa entre particiones.")
    pos = [set(position_key(frame, position_columns)) for frame in [train, val, test]]
    return {
        "rows": {"train": len(train), "val": len(val), "test": len(test)},
        "positions": {"train": len(pos[0]), "val": len(pos[1]), "test": len(pos[2])},
        "position_overlap": {
            "train_val": len(pos[0] & pos[1]),
            "train_test": len(pos[0] & pos[2]),
            "val_test": len(pos[1] & pos[2]),
        },
    }


## 6. Preparación completa de UJIIndoor


In [6]:
def prepare_uji_dataset(
    data_directories: Sequence[Path],
    output_dir: Path,
    building_id: Optional[int] = 1,
    floors: Optional[Sequence[int]] = (0, 1),
    k_values: Sequence[int] = tuple(range(2, 11)),
    val_size: float = 0.15,
    seed: int = SEED,
    metric_mode: str = "epsg3857_to_epsg25830",
    gate_estimators: int = 200,
    test_known_clients_only: bool = False,
) -> Dict[str, object]:
    dirs = [Path(p) for p in data_directories]
    train_path = find_file_case_insensitive("TrainingData.csv", dirs)
    test_path = find_file_case_insensitive("ValidationData.csv", dirs)
    official_train = normalize_columns(pd.read_csv(train_path))
    official_test = normalize_columns(pd.read_csv(test_path))

    def apply_filter(frame: pd.DataFrame) -> pd.DataFrame:
        mask = np.ones(len(frame), dtype=bool)
        if building_id is not None:
            mask &= pd.to_numeric(frame["BUILDINGID"], errors="coerce").eq(building_id).to_numpy()
        if floors is not None:
            mask &= pd.to_numeric(frame["FLOOR"], errors="coerce").isin(list(floors)).to_numpy()
        return frame.loc[mask].reset_index(drop=True)

    official_train = apply_filter(official_train)
    official_test = apply_filter(official_test)
    if official_train.empty or official_test.empty:
        raise ValueError("El filtro de edificio/planta dejo train o test vacio.")
    required = ["LONGITUDE", "LATITUDE", "PHONEID", "BUILDINGID", "FLOOR"]
    missing = [c for c in required if c not in official_train.columns]
    if missing:
        raise ValueError(f"Faltan columnas UJIIndoorLoc: {missing}")

    pos_cols = ["BUILDINGID", "FLOOR", "LONGITUDE", "LATITUDE"]
    train, val = split_uji_training_by_position(official_train, pos_cols, val_size, seed)
    test = official_test.copy().reset_index(drop=True)

    known_clients = set(
        pd.concat([train["PHONEID"], val["PHONEID"]], ignore_index=True)
        .astype(str)
        .tolist()
    )
    test_clients_before = set(test["PHONEID"].astype(str).tolist())
    excluded_test_clients = sorted(test_clients_before - known_clients)
    test_rows_before = len(test)
    if test_known_clients_only:
        test = test[test["PHONEID"].astype(str).isin(known_clients)].reset_index(drop=True)
        if test.empty:
            raise ValueError(
                "El test UJIIndoorLoc quedo vacio tras conservar solo dispositivos "
                "presentes en train/validacion."
            )
    test_client_filter = {
        "enabled": bool(test_known_clients_only),
        "known_train_val_clients": sorted(known_clients),
        "excluded_test_clients": excluded_test_clients if test_known_clients_only else [],
        "rows_before": int(test_rows_before),
        "rows_after": int(len(test)),
        "rows_removed": int(test_rows_before - len(test)),
    }

    for frame in [train, val, test]:
        frame["CLIENT_ID"] = frame["PHONEID"].astype(str)
        frame["FLOOR_LABEL"] = pd.to_numeric(
            frame["FLOOR"], errors="raise"
        ).astype(int)
    train, val, test = [add_uji_metric_targets(frame, metric_mode) for frame in [train, val, test]]
    prefix = "uji_b1_f01" if building_id == 1 and tuple(floors or ()) == (0, 1) else "uji_custom"
    paths = save_base_splits(train, val, test, output_dir, prefix)
    saved = read_base_splits(output_dir, prefix)
    rssi = detect_rssi_columns(saved["train"])
    routes, route_diag = fit_rssi_routing(
        saved["train"], saved["val"], saved["test"], rssi, k_values, seed, gate_estimators
    )
    route_paths = save_routing(routes, route_diag, output_dir, prefix)
    diagnostics = validate_splits(saved["train"], saved["val"], saved["test"], pos_cols)
    diagnostics["clients"] = {
        split: sorted(frame["CLIENT_ID"].astype(str).unique().tolist())
        for split, frame in saved.items()
    }
    diagnostics["floors"] = {
        split: sorted(
            pd.to_numeric(frame["FLOOR_LABEL"], errors="raise")
            .astype(int)
            .unique()
            .tolist()
        )
        for split, frame in saved.items()
    }
    diagnostics["unseen_test_clients"] = sorted(
        set(saved["test"]["CLIENT_ID"].astype(str))
        - (
            set(saved["train"]["CLIENT_ID"].astype(str))
            | set(saved["val"]["CLIENT_ID"].astype(str))
        )
    )
    diagnostics["test_client_filter"] = test_client_filter
    diagnostics["metric_crs"] = sorted(saved["train"]["METRIC_CRS"].astype(str).unique().tolist())
    results = {
        "prefix": prefix,
        "files": {**{k: str(v) for k, v in paths.items()}, "routes": str(route_paths[0])},
        "diagnostics": diagnostics,
        "routing": route_diag.to_dict("records"),
    }
    output_dir = Path(output_dir)
    with (output_dir / f"{prefix}_preparation_report.json").open("w", encoding="utf-8") as handle:
        json.dump(results, handle, indent=2, ensure_ascii=False)
    return results


## 7. Configuración y ejecución


In [7]:
# Se necesitan los CSV originales completos de UJIIndoorLoc. El ZIP recibido
# estaba truncado en la parte del CSV grande, por lo que no se usa ese fragmento.
DATA_DIRS = [
    Path.cwd(),
    ROOT / "UJIIndoor",
    ROOT / "Dataset",
    ROOT.parent,
    ROOT.parent / "Dataset",
    Path("/mnt/data"),
]
OUTPUT_DIR = ROOT / "prepared" / "UJIIndoor"

BUILDING_ID = 1
FLOORS = (0, 1)
K_VALUES = list(range(2, 11))
VALIDATION_SIZE = 0.15
SEED = 42
GATE_TREES = 200
TEST_KNOWN_DEVICES_ONLY = True

# Recomendado para los valores (-7500, 4.86e6) del UJI original.
# Alternativa si confirmas que ya son coordenadas métricas locales:
# UJI_METRIC_MODE = "projected_as_metres"
UJI_METRIC_MODE = "epsg3857_to_epsg25830"

report = prepare_uji_dataset(
    data_directories=DATA_DIRS,
    output_dir=OUTPUT_DIR,
    building_id=BUILDING_ID,
    floors=FLOORS,
    k_values=K_VALUES,
    val_size=VALIDATION_SIZE,
    seed=SEED,
    metric_mode=UJI_METRIC_MODE,
    gate_estimators=GATE_TREES,
    test_known_clients_only=TEST_KNOWN_DEVICES_ONLY,
)
PREFIX = report["prefix"]
print("Datos preparados en:", OUTPUT_DIR)
print("Prefijo para los modelos:", PREFIX)


/tmp/ipykernel_1427583/3330427122.py:39: RuntimeWarning: pyproj no esta instalado: se usa correccion local cos(lat). Instala pyproj para obtener EPSG:25830 exacto.
  warnings.warn(


Datos preparados en: /home/coder/Indoor/Notebooks/prepared/UJIIndoor
Prefijo para los modelos: uji_b1_f01


## 8. Auditoría final


In [8]:
print("\n=== Auditoría de particiones ===")
display(pd.DataFrame([report["diagnostics"]["rows"]], index=["filas"]))
display(pd.DataFrame([report["diagnostics"]["positions"]], index=["posiciones"]))
display(pd.DataFrame([report["diagnostics"]["position_overlap"]], index=["solapamiento"]))
print("Clientes de test ausentes en train/validación:", report["diagnostics"]["unseen_test_clients"])
print("Filtro de dispositivos de test:", report["diagnostics"]["test_client_filter"])
print("Plantas por partición:", report["diagnostics"]["floors"])
print("Sistema métrico:", report["diagnostics"]["metric_crs"])

print("\n=== Calidad del enrutador RSSI ===")
display(pd.DataFrame(report["routing"]))

assert report["diagnostics"]["position_overlap"]["train_val"] == 0
assert report["diagnostics"]["unseen_test_clients"] == []
print("\nOK: validación no comparte posiciones con train y test solo contiene dispositivos conocidos.")



=== Auditoría de particiones ===


,train,val,test
filas,2489,363,67


,train,val,test
posiciones,115,21,67


,train_val,train_test,val_test
solapamiento,0,1,0


Clientes de test ausentes en train/validación: []
Filtro de dispositivos de test: {'enabled': True, 'known_train_val_clients': ['13', '14', '17', '6', '7'], 'excluded_test_clients': ['0', '12', '15', '2', '20', '21', '4', '5', '9'], 'rows_before': 173, 'rows_after': 67, 'rows_removed': 106}
Plantas por partición: {'train': [0, 1], 'val': [0, 1], 'test': [0, 1]}
Sistema métrico: ['LOCAL_GROUND_APPROX_COS_LAT_0.76612616']

=== Calidad del enrutador RSSI ===


,N_CLUSTERS,VAL_GATE_ACCURACY,TEST_GATE_ACCURACY_DIAGNOSTIC_ONLY,VAL_MEAN_CONFIDENCE,TEST_MEAN_CONFIDENCE,N_TRAIN_POSITIONS
0,2,0.997245,0.910448,0.987727,0.916354,106
1,3,1.000000,0.955224,0.986184,0.868655,106
2,4,1.000000,0.985075,0.979577,0.869936,106
3,5,1.000000,0.985075,0.967846,0.871566,106
4,6,0.994490,0.865672,0.917726,0.792313,106
5,7,0.986226,0.731343,0.912652,0.770108,106
6,8,0.966942,0.701493,0.910869,0.740904,106
7,9,0.944904,0.626866,0.839260,0.647129,106
8,10,0.768595,0.671642,0.733065,0.607792,106



OK: validación no comparte posiciones con train y test solo contiene dispositivos conocidos.


`TEST_GATE_ACCURACY_DIAGNOSTIC_ONLY` ayuda a explicar el error, pero nunca se
usa para escoger K. Los notebooks de modelos escogen hiperparámetros y K con
RMSE de validación para coordenadas y con accuracy de validación para planta.
Test se consulta una sola vez después de fijar cada configuración.
